# What is Arrow Flight - a crash course on GRPC and Protobuf

*Arrow Flight is an RPC framework for high-performance data services based on Arrow data, and is built on top of gRPC and the IPC format.*
> https://arrow.apache.org/docs/format/Flight.html

In essence, Arrow Flight is a GRPC-based standard that uses Apache Arrow as the data interchange format. At its core, Arrow Flight is defined as a large Protobuf file.
To understand Arrow Flight, we need to do a quick digression into Protobuf and GRPC, since that is the foundation of Arrow Flight.



![Protobuf Logo](images/protobuf-logo.png)


## What is Protobuf?
Protobuf is a message format that is described using a protobuf file. That file can then be compiled into different languages.

A protobuf file looks something like this:

```protobuf
message Person {
    string name = 1;
    int32 id = 2;
    string email = 3;
}
```

A `message` defined by some fields of different datatypes. 

In [9]:
from gen import person_pb
import json

person = person_pb.Person(name="Anders", id=3, email="test@test.com")

In [2]:
binary_rep = person.to_binary()
binary_rep

b'\n\x06Anders\x10\x03\x1a\rtest@test.com'

In [8]:
len(binary_rep)

25

In [11]:
json_rep = json.dumps({"name": "Anders", "id": 3, "email": "test@test.com"}).encode()
json_rep

b'{"name": "Anders", "id": 3, "email": "test@test.com"}'

In [12]:
print(f"Size of Protobuf message: {len(binary_rep)}\nSize of JSON: {len(json_rep)}")

Size of Protobuf message: 25
Size of JSON: 53


In [13]:
person_pb.Person.from_binary(binary_rep)

Person(name='Anders', id=3, email='test@test.com')

## What is a GRPC service

A GRPC service is a way of defining Remote Procedure Calls in Protobuf, essentially sending a command to be executed on the server, as if it was running locally.
Old idea, but GRPC is currently the best way to do it.

![grpc](images/grpc.svg)

in our protobuf file, we can add a `service` definition, alongside some more messages

```protobuf
message GetPersonRequest {
    int32 id = 1;
}

message GetPersonResponse {
    Person person = 1;
}

message GetPersonsResponse {
    repeated Person persons = 1;
}

service PersonService {
    rpc GetUser(GetPersonRequest) returns (GetPersonResponse);
    rpc GetStreamUsers(GetPersonRequest) returns (stream GetPersonResponse);
    rpc SendStreamUsers(stream GetPersonRequest) returns (GetPersonsResponse);
    rpc SendAndGetStreamUsers(stream GetPersonRequest) returns (stream GetPersonResponse);
}
```

One defining feature of GRPC, is that it uses HTTP/2, which allows it to include bidirectional streaming capabilities

In [14]:
from gen.person_pb_grpc import PersonServiceServiceSync, GetPersonRequest, GetPersonResponse, PersonServiceClientSync, GetPersonsResponse
from gen.person_pb import Person
import grpc
from concurrent import futures
from typing import Iterator

To illustrate, we can implement this simple Person Service for our 4 different RPC methods to demonstrate.

GRPC supports both sync and async services, but here we use the Sync version for simplicity. 

Inherit from the autogenerated `PersonServiceService` and implement the methods.

> PS) Unary is just GRPC speak for "not-streaming"

In [15]:
my_db = {
    1: Person(id=1, name="Vincent Van Gogh", email="v.vangogh@example.nl"),
    2: Person(id=2, name="Rembrandt van Rijn", email="r.vanrijn@example.nl"),
    3: Person(id=3, name="Antoni van Leeuwenhoek", email="a.vanleeuwenhoek@example.nl")
}
class MyService(PersonServiceServiceSync):
    def __init__(self, db_url: str):
        self.db_url = db_url

    # Unary - Unary
    def get_user(self, request: GetPersonRequest, context: grpc.ServicerContext) -> GetPersonResponse:
        return GetPersonResponse(person=my_db[request.id])

    # Unary - Stream
    def get_stream_users(self, request: GetPersonRequest, context: grpc.ServicerContext) -> Iterator[GetPersonResponse]:
        for person in my_db.values():
            yield GetPersonResponse(person=person)

    # Stream - Unary
    def send_stream_users(self, request: Iterator[GetPersonRequest], context: grpc.ServicerContext) -> GetPersonsResponse:
        people = [my_db[r.id] for r in request]
        return GetPersonsResponse(persons=people)

    # Stream - Stream
    def send_and_get_stream_users(self, request: Iterator[GetPersonRequest], context: grpc.ServicerContext) -> Iterator[GetPersonResponse]:
        for r in request:
            yield GetPersonResponse(person=my_db[r.id])
    

Now we just need to start the server - since we're using the Sync version, we need to pass a threadpool of workers

In [16]:
def serve() -> grpc.Server:
    my_service = MyService(db_url="mydemodb")
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    my_service.add_to_server(server)
    server.add_insecure_port("[::]:5000")
    server.start()
    return server

In [17]:
# Ensure we stop any existing servers before starting a new one
if 'server' in dir() and server is not None:
    server.stop(grace=0).wait()
server = serve()

We can now connect a Client to our Server. 

GRPC is designed to work with TLS, so we need to explicitly tell it we're using non-encrypted traffic. Don't do this in production!

We create a channel and create a Client, also autogenerated from the Protobuf file. This client only knows what methods its supposed to have, and how to serialize and deserialize the data from the server.

In [18]:
with grpc.insecure_channel("localhost:5000") as channel:
    client = PersonServiceClientSync(channel)
    response = client.get_user(GetPersonRequest(id=1))

response.person

Person(name='Vincent Van Gogh', id=1, email='v.vangogh@example.nl')

In [19]:
channel = grpc.insecure_channel("localhost:5000")
client = PersonServiceClientSync(channel)
response = client.get_stream_users(GetPersonRequest(id=0))
for r in response:
    print(r.person)

Person(name='Vincent Van Gogh', id=1, email='v.vangogh@example.nl')
Person(name='Rembrandt van Rijn', id=2, email='r.vanrijn@example.nl')
Person(name='Antoni van Leeuwenhoek', id=3, email='a.vanleeuwenhoek@example.nl')


In [20]:
def _send_users() -> Iterator[GetPersonRequest]:
    for user_id in [1, 2]:
        yield GetPersonRequest(id=user_id)
response = client.send_stream_users(_send_users())
response.persons

[Person(name='Vincent Van Gogh', id=1, email='v.vangogh@example.nl'),
 Person(name='Rembrandt van Rijn', id=2, email='r.vanrijn@example.nl')]

In [21]:
response = client.send_and_get_stream_users(_send_users())
for r in response:
    print(r.person)

Person(name='Vincent Van Gogh', id=1, email='v.vangogh@example.nl')
Person(name='Rembrandt van Rijn', id=2, email='r.vanrijn@example.nl')


In [22]:
channel.close()

That was a crash course on Protobuf and GRPC!